In [1]:
import os, json, pathlib, re, random, time, ast
from typing import List, Dict, Optional
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

# LangChain / FAISS
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings  # keep to reuse your existing index

# ---- Gemma / Google GenAI ----
from google import genai
from google.genai import types

In [2]:
# Access API
load_dotenv()
MODEL_NAME  = "gemma-3-27b-it"
TEMPERATURE = 1.0
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Gemma model:", MODEL_NAME)

# Load files
INDEX_DIR = pathlib.Path("faiss_disaster_information_index")  # 复用原索引
src_file  = "../data/disaster_description.csv"
out_file  = "extracted_Gemma.csv"
df = pd.read_csv(src_file)
# df = df.head(50)

GOOGLE_API_KEY loaded: True
Gemma model: gemma-3-27b-it


# FAISS Vectorstore

In [4]:
DISASTER_CLASSIFICATION = json.load(open("../data/classification_document.json"))

In [5]:
MAGNITUDE_INFORMATION = json.load(open("../data/magnitude_document.json"))

In [6]:
FIELD_SCHEMA = json.load(open("../data/schema_document.json"))

In [7]:
EXAMPLES = pathlib.Path("../data/Examples.txt").read_text(encoding="utf-8")

In [8]:
disaster_docs = []
groups = DISASTER_CLASSIFICATION["Disaster Group"]
for group_name, meta in groups.items():
        desc  = meta.get("Description", "")
        type = meta.get("Disaster Types", {})

        block = f"### Disaster Group: {group_name}\n{desc}\n\nDisaster Types:\n"
        for t_name, t_desc in type.items():
            block += f"- {t_name}: {t_desc}\n"

        doc = Document(
            page_content = block.strip(),
            metadata     = {"group": group_name, "level": "G1"}
        )
        disaster_docs.append(doc)

In [9]:
magnitude_docs = []
for dtype, info in MAGNITUDE_INFORMATION.items():
    block = (
        f"### Magnitude Guidance\n"
        f"Disaster Type: {dtype}\n"
        f"Property      : {info['property']}\n"
        f"Unit          : {info['unit']}"
    )

    doc = Document(
        page_content = block,
        metadata     = {"type" : dtype, "level": "M1" }
    )
    magnitude_docs.append(doc)

In [10]:
fields_docs = []
for section_name, fields in FIELD_SCHEMA.items():
    for fname, meta in fields.items():
        block = (
            f"### Field: {fname}\n"
            f"Section      : {section_name}\n"
            f"Type         : {meta['Type']}\n"
            f"Description  : {meta.get('Description', meta.get('Rescription',''))}"
        )

        doc = Document(
            page_content = block,
            metadata = {"section": section_name, "field"  : fname, "level"  : "F0"
                }
        )
        fields_docs.append(doc)

In [11]:
examples_docs = []
block = re.split(r"Example\s+\d+:\s*", EXAMPLES)[1:]

for idx, chunk in enumerate(block, start=1):
    doc = Document(
        page_content = chunk.strip(),
        metadata     = {"example_id": idx, "level": "E1"}
    )
    examples_docs.append(doc)

In [12]:
if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(
        str(INDEX_DIR),
        OpenAIEmbeddings(model="text-embedding-3-small"),
        allow_dangerous_deserialization=True 
    )
else:
    print("Building FAISS index from scratch … ")
    
    embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(disaster_docs, embeddings)
    vectorstore.add_documents(magnitude_docs)
    vectorstore.add_documents(fields_docs)
    vectorstore.add_documents(examples_docs)
    vectorstore.save_local(str(INDEX_DIR))

print(f"✅  Vectorstore ready, ntotal = {vectorstore.index.ntotal}")

✅  Vectorstore ready, ntotal = 41


# Prompt

In [14]:
OUTPUT_FIELDS = {
  "Event": "fields name",
  "Geographical Information": "fields name",
  "Effect": "fields name"
}

In [15]:
OUTPUT_VALUES = {
  "Event": {"fields": "values"},
  "Geographical Information": {"fields": "values"},
  "Effect": {"fields": "values"}
}

In [16]:
def restructure_example(doc: Document):
    
    txt = doc.page_content

    # description
    e_desc = re.search(r"Description:\s*(.*?)\s*Fields output:", txt, re.S)
    if not e_desc:
        raise ValueError(f"Did not find description, example_id={doc.metadata.get('example_id')}")
    description = e_desc.group(1).strip()

    # fields / values
    e_fields = re.search(r"Fields output:\s*(\{.*?\})\s*Values output:", txt, re.S)
    e_values = re.search(r"Values output:\s*(\{.*\})\s*$", txt, re.S)
    
    if not (e_fields and e_values):
        raise ValueError(f"Did not find description fields/values, example_id={doc.metadata.get('example_id')}")

    example_fields  = ast.literal_eval(e_fields.group(1).strip())
    example_values  = ast.literal_eval(e_values.group(1).strip())

    # restructure
    stage1 = {
        "description":    description,
        "output_fields":  example_fields,
    }
    stage2 = {
        "description":    description,
        "output_values":  example_values,
    }
    return stage1, stage2


In [17]:
def fields_extraction_prompt (description: str, example1: Document, example2: Document) -> str:
    
    stage1_dict1, _ = restructure_example(example1)
    stage1_dict2, _ = restructure_example(example2)
    
    prompt = f"""
        Please refer FIELD_SCHEMA first, then list only fields that are marked as 'Required' or are explicitly mentioned in DISASTER_DESCRIPTION. 
        Output must contain field names only.
        Finally return a single JSON object that conforms precisely to OUTPUT_FIELDS.
        Below there are two reference EXAMPLE1_STAGE1 and EXAMPLE2_STAGE1 
        for one disaster's description and its output fields.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        OUTPUT_FIELDS:
        {json.dumps(OUTPUT_FIELDS, ensure_ascii=False)}
    
        DISASTER_DESCRIPTION:
        {description}

        EXAMPLE_STAGE1:
        {json.dumps(stage1_dict1, ensure_ascii=False, indent=4)}

        EXAMPLE2_STAGE1:
        {json.dumps(stage1_dict2, ensure_ascii=False, indent=4)}

    """
    return prompt

In [18]:
def values_extraction_prompt(
    description: str, 
    fields_docs: List[Document], 
    extracted_fields: str,
    example: Document,
    categorical_values: Optional[Document]
) -> str:
    
    PROMPT = """
    Please extract all information regarding only the fields in EXTRACTED_FIELDS 
    from text DISASTER_DESCRIPTION (refering the information in FIELD_SCHEMA and using only 
    the categorical options in CATEGORICAL_VALUES). 
    Finally return a single JSON object that conforms precisely to OUTPUT_VALUES.
    Below there are two reference EXAMPLE1_STAGE2 and EXAMPLE2_STAGE2 
    for one disaster's description and its output values.
"""

    parts = ["FIELD_SCHEMA:"]
    parts += [d.page_content for d in fields_docs]

    parts += ["CATEGORICAL_VALUES:"]
    parts += [d.page_content for d in categorical_values] 

    parts += [
        "OUTPUT_VALUES:",
        json.dumps(OUTPUT_VALUES, ensure_ascii=False),
        "DISASTER_DESCRIPTION:",
        description,
        "EXTRACTED_FIELDS:",
        json.dumps(extracted_fields, ensure_ascii=False)
    ]

    _, stage2_dict= restructure_example(example)
    parts += ["EXAMPLE1_STAGE2:",
              json.dumps(stage2_dict, ensure_ascii=False, indent=4)]
    
    return PROMPT + "\n\n" + "\n\n".join(parts)


# Implement

In [20]:
def extract_json_from_text(text: str):
    """从自由文本中尽可能提取第一个合法 JSON（对象或数组）"""
    if not text:
        return None
    text = text.strip()

    # 1) 直接就是 JSON
    if text and text[0] in "[{":
        try:
            return json.loads(text)
        except Exception:
            pass

    # 2) 三引号代码块 ```json ... ```
    m = re.search(r"```(?:json|JSON)?\s*([\s\S]*?)\s*```", text)
    if m:
        candidate = m.group(1).strip()
        try:
            return json.loads(candidate)
        except Exception:
            # 有时代码块里还有解释文字，再用括号匹配法
            parsed = _brace_scan(candidate)
            if parsed is not None:
                return parsed

    # 3) 括号扫描（支持字符串、转义），找到第一个完整的 {} 或 []
    parsed = _brace_scan(text)
    if parsed is not None:
        return parsed

    return None

In [21]:
def _parse_retry_delay_seconds(err) -> int:
    """从异常对象里抓取 RetryInfo.retryDelay，形如 '52s' 或 '1.2s'"""
    # 优先从 e.args[0] 里拿结构化字典
    if getattr(err, "args", None) and isinstance(err.args[0], dict):
        try:
            details = err.args[0]["error"]["details"]
            for d in details:
                if d.get("@type", "").endswith("RetryInfo"):
                    v = d.get("retryDelay", "0s")
                    m = re.match(r"(\d+(?:\.\d+)?)s", v)
                    if m:
                        return max(1, int(float(m.group(1))))
        except Exception:
            pass
    # 再从字符串里兜底解析
    m = re.search(r"retryDelay['\"]?\s*:\s*['\"]?(\d+(?:\.\d+)?)s", str(err))
    if m:
        return max(1, int(float(m.group(1))))
    return 15  # 没拿到就退而求其次

In [22]:
def call_llm(prompt: str, max_retries: int = 4):
    backoff = 2 
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                ),
            )
            text = getattr(resp, "text", None)
            parsed = extract_json_from_text(text)  # ← 用你/我上条消息给的解析器
            if parsed is not None:
                return parsed
            head = (text or "")[:200].replace("\n", " ")
            print(f"[ParseFail attempt={attempt}] head=", head)

        except Exception as e:
            # 配额/速率类错误：按 retryDelay 等待
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait_s = _parse_retry_delay_seconds(e)
                # 加一点抖动，避免“同一分钟边界”撞车
                wait_s += random.randint(1, 3)
                print(f"[429] waiting {wait_s}s then retry...")
                time.sleep(wait_s)
                continue

            print(f"LLM Error (attempt {attempt}):", e)

        # 其他失败：指数退避
        time.sleep(backoff + random.random())
        backoff = min(backoff * 2, 16)

    return None

In [23]:
def process_record(description: str) -> dict:
    if not isinstance(description, str):
        return None 

    # examples
    e1_docs = vectorstore.similarity_search(
        description, 
        k=2,
        filter={"level": "E1"}
    )
    if len(e1_docs) < 2:
        used_ids = {d.metadata.get("example_id") for d in e1_docs}
        candidates = [d for d in examples_docs if d.metadata.get("example_id") not in used_ids]
        while len(e1_docs) < 2 and candidates:
            e1_docs.append(candidates.pop())
    
    # Stage1: fields
    p1 = fields_extraction_prompt(description, e1_docs[0], e1_docs[1])
    r1 = call_llm(p1)

    if not isinstance(r1, dict) or r1 is None:
        return None
    
    # Stage2: values
    # 2-1: fields_docs
    if not isinstance(r1, dict):
        print("The type of r1 is not dictionary:", r1)
        return None
    try:
        field_list = [fname for section in r1.values() for fname in section]
    except Exception as e:
        print("The values of r1 are not lists", r1)
        return None        
    
    f0_docs = []
    for fname in field_list:
        doc = vectorstore.similarity_search(
            description,
            k=1,
            filter={"level": "F0", "field": fname}
        )
        if doc:
            f0_docs.append(doc[0])
        else:
            # fallback
            field_doc_map = {d.metadata["field"]: d.page_content for d in fields_docs}
            f0_docs.append(
                Document(
                    page_content=field_doc_map.get(fname, ""),
                    metadata={"level": "F0", "field": fname}
                )
            )
    
    # 2-2: categorical values
    g1_docs = vectorstore.similarity_search(
        description,
        k=2,
        filter={"level": "G1"}
    )
    
    m1_doc = None
    if {"Magnitude", "Magnitude Scale"} & set(field_list):
        hits = vectorstore.similarity_search(
            description,
            k=1,
            filter={"level": "M1"}
        )
        m1_doc = hits[0] if hits else None

    categorical_values = g1_docs + ([m1_doc] if m1_doc else [])


    
    # 2-3: extraction
    p2 = values_extraction_prompt(
        description     = description,
        fields_docs      = f0_docs,
        extracted_fields= r1,
        example = e1_docs[0],
        categorical_values = categorical_values
    )
    r2 = call_llm(p2)
    
    if r2 is None:
        return None
    
    return {
        "Gemma_fields": json.dumps(r1, ensure_ascii=False),
        "Gemma_values": json.dumps(r2, ensure_ascii=False)
    }


In [24]:
results = []
for desc in tqdm(df["description"], desc="Processing"):
    out = process_record(desc)
    if out is None:
        results.append({"Gemma_fields": None, "Gemma_values": None})
    else:
        results.append(out)

df_out = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
df_out.to_csv(out_file, index=False)
print(f"✅Saved to {out_file}")

Processing:   0%|                            | 4/1588 [00:39<4:05:55,  9.32s/it]

[429] waiting 36s then retry...


Processing:   1%|▏                           | 8/1588 [01:55<6:13:51, 14.20s/it]

[429] waiting 22s then retry...


Processing:   1%|▏                          | 12/1588 [02:56<5:25:55, 12.41s/it]

...

[429] waiting 33s then retry...


Processing: 100%|████████████████████████▉| 1583/1588 [6:15:41<00:51, 10.21s/it]

[429] waiting 32s then retry...


Processing: 100%|█████████████████████████| 1588/1588 [6:16:45<00:00, 14.23s/it]

✅Saved to extracted_Gemma.csv
